# Phase B — Backbone Investigation (E02–E05) on MVTec LOCO AD (Combined)

Compares **4 modern backbones** integrated into the PUAD framework against verified **E01 Original PUAD** (`87.56%` Combined Mean / `83.30%` Pooled Raw):
- **E02**: `ResNet-18` (residual inductive bias)
- **E03**: `DenseNet-121` (dense feature reuse)
- **E04**: `EfficientNet-B0` (compound-scaled MBConv efficiency)
- **E05**: `ConvNeXt-Tiny` (modern large-kernel pure-CNN)

All 4 experiments use the **exact same controlled protocol**:
- Pooled 5-category `train/good` (`1,778`), `validation/good` (`305`), logical-only test (`1,136`, **0** structural).
- Frozen ImageNet-pretrained backbone teacher (`{C3, C4, C5}` → `384×32×32`) + randomly initialized trainable student + 1×1 global AutoEncoder + Ledoit-Wolf Mahalanobis + validation z-score fusion.
- `batch_size=8`, `AdamW`, `lr=3e-4` (cosine to `1e-5`), `200 epochs`, `patience=20`, hardware-native AMP (`BF16` on A100/L4, `FP16+GradScaler` on T4), auto-resumable & synced to Google Drive every epoch.

In [ ]:
#@title 1. Section 3 — Hardware, Runtime, RAM & Disk Audit
import os, platform, shutil, sys, torch

print('=== COLAB SESSION AUDIT (PHASE B) ===')
print('Python        :', sys.version.split()[0])
print('PyTorch       :', torch.__version__, '| CUDA:', torch.version.cuda)
print('CUDA Available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU assigned! Switch Runtime -> Change runtime type -> GPU.'
props = torch.cuda.get_device_properties(0)
native_bf16 = props.major >= 8 and torch.cuda.is_bf16_supported()
print(f'GPU Model     : {props.name} (CC {props.major}.{props.minor})')
print(f'GPU VRAM      : {props.total_memory / (1024**3):.2f} GiB | Native BF16: {native_bf16}')
import psutil
vm = psutil.virtual_memory()
print(f'System RAM    : {vm.available/(1024**3):.1f} GB free / {vm.total/(1024**3):.1f} GB total')
disk = shutil.disk_usage('/content')
print(f'Local SSD     : {disk.free/(1024**3):.1f} GB free / {disk.total/(1024**3):.1f} GB total')

In [ ]:
#@title 2. Mount Drive, Pull Latest Code & Ensure Local SSD Datasets
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT_DIR = '/content/drive/MyDrive/DL_Endsem_Team11'
DRIVE_LOCO_TAR    = f'{DRIVE_PROJECT_DIR}/data/mvtec_loco_anomaly_detection.tar.xz'
DRIVE_IMNET_ZIP   = f'{DRIVE_PROJECT_DIR}/data/imagenet_penalty_10k.zip'
DRIVE_TEACHER_PT  = f'{DRIVE_PROJECT_DIR}/weights/teacher_pdn_s.pt'
DRIVE_RESULTS_DIR = f'{DRIVE_PROJECT_DIR}/runs'
GITHUB_REPO_URL   = 'https://github.com/Chimera418/dl-endsem-logical-ad.git'

REPO_DIR = '/content/Endsem'
if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', GITHUB_REPO_URL, REPO_DIR], check=True)

!pip install -q timm thop einops pyyaml psutil tqdm
%cd /content/Endsem

LOCAL_LOCO = '/content/data/mvtec_loco_anomaly_detection'
LOCAL_IMNET = '/content/data/imagenet_penalty'
os.makedirs('/content/data', exist_ok=True)
if not os.path.isdir(os.path.join(LOCAL_LOCO, 'breakfast_box')):
    os.makedirs(LOCAL_LOCO, exist_ok=True)
    !tar -xf "$DRIVE_LOCO_TAR" -C "$LOCAL_LOCO"
if not os.path.isdir(LOCAL_IMNET):
    !unzip -q "$DRIVE_IMNET_ZIP" -d /content/data/

os.environ['ENDSEM_LOCO_ROOT'] = LOCAL_LOCO
os.environ['ENDSEM_IMAGENET_ROOT'] = LOCAL_IMNET
os.environ['ENDSEM_TEACHER_PATH'] = DRIVE_TEACHER_PT
!python scripts/verify_env.py

In [ ]:
#@title 3. Run E01 (200-Epoch Unified Control) — Original PUAD
!python scripts/run_experiment.py --config configs/E01_puad.yaml --mode train --drive-sync-dir "$DRIVE_RESULTS_DIR"

In [ ]:
#@title 4. Run E02 — ResNet-18
!python scripts/run_experiment.py --config configs/E02_resnet18.yaml --mode train --drive-sync-dir "$DRIVE_RESULTS_DIR"

In [ ]:
#@title 5. Run E03 — DenseNet-121
!python scripts/run_experiment.py --config configs/E03_densenet121.yaml --mode train --drive-sync-dir "$DRIVE_RESULTS_DIR"

In [ ]:
#@title 6. Run E04 — EfficientNet-B0
!python scripts/run_experiment.py --config configs/E04_efficientnet_b0.yaml --mode train --drive-sync-dir "$DRIVE_RESULTS_DIR"

In [ ]:
#@title 7. Run E05 — ConvNeXt-Tiny
!python scripts/run_experiment.py --config configs/E05_convnext_tiny.yaml --mode train --drive-sync-dir "$DRIVE_RESULTS_DIR"

In [ ]:
#@title 8. Aggregate & Compare E01–E05 (Phase B Summary Table)
!python scripts/aggregate_results.py --dirs experiments "$DRIVE_RESULTS_DIR" --out-dir "$DRIVE_RESULTS_DIR/phaseB_summary"